# 掩膜可视化演示

本笔记本演示 `mask` 模块中包含的各种掩膜类型的生成和可视化。

In [ ]:
# 导入必要的库
import sys
import numpy as np
import matplotlib.pyplot as plt
import os
import sys

# 获取当前 notebook 所在目录
current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()

# 添加项目根目录（即 src 的上级）
project_root = os.path.abspath(os.path.join(current_dir, "../.."))
if project_root not in sys.path:
    sys.path.append(project_root)

print("✅ 已添加项目根目录到 sys.path:", project_root)

# 设置绘图样式
plt.rcParams['figure.figsize'] = (15, 10)

# -------------------------------
# ✅ 全局字体设置
# -------------------------------
plt.rcParams['font.family'] = ['STHeiti', 'Arial']
plt.rcParams['axes.unicode_minus'] = False             # 正确显示负号
plt.rcParams['font.size'] = 14                         # 默认字体大小
plt.rcParams['axes.titlesize'] = 16                    # 标题字体大小
plt.rcParams['axes.labelsize'] = 14                    # 坐标轴标签字体大小
plt.rcParams['xtick.labelsize'] = 12                   # x轴刻度字体大小
plt.rcParams['ytick.labelsize'] = 12                   # y轴刻度字体大小
plt.rcParams['legend.fontsize'] = 12                   # 图例字体大小
plt.rcParams['figure.titlesize'] = 18                  # 整个图形标题字体大小

from src.utils.mask import CodedAperture, MultiLensArray, PhaseContour, FresnelZoneAperture, RandomBinaryMask


## 1. CodedAperture - 编码孔径掩膜 (FlatCam)

编码孔径掩膜使用 MURA 或 MLS 方法生成伪随机二值模式。

In [ ]:
# 创建 MLS 方法的编码孔径掩膜
coded_aperture_mls = CodedAperture(
    method="MLS",
    n_bits=8,
    resolution=(256, 256),
    feature_size=1e-5,  # 10 微米像素
    distance_sensor=4e-3,  # 4mm 到传感器的距离
)

# 绘制掩膜
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
coded_aperture_mls.plot(ax=axes[0])
axes[0].set_title("CodedAperture - MLS 方法")

# 显示 PSF
if coded_aperture_mls.psf is not None:
    psf_rgb = coded_aperture_mls.psf / coded_aperture_mls.psf.max()
    axes[1].imshow(psf_rgb)
    axes[1].set_title("对应的 PSF (点扩散函数)")
    axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"掩膜分辨率: {coded_aperture_mls.resolution}")
print(f"掩膜尺寸: {coded_aperture_mls.size * 1e3} mm")
print(f"掩膜形状: {coded_aperture_mls.mask.shape}")

In [ ]:
# 创建 MURA 方法的编码孔径掩膜 (需要质数)
coded_aperture_mura = CodedAperture(
    method="MURA",
    n_bits=11,  # 使用质数
    resolution=(256, 256),
    feature_size=1e-5,
    distance_sensor=4e-3,
)

# 绘制掩膜
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
coded_aperture_mura.plot(ax=axes[0])
axes[0].set_title("CodedAperture - MURA 方法")

# 显示 PSF
if coded_aperture_mura.psf is not None:
    psf_rgb = coded_aperture_mura.psf / coded_aperture_mura.psf.max()
    axes[1].imshow(psf_rgb)
    axes[1].set_title("对应的 PSF (点扩散函数)")
    axes[1].axis('off')

plt.tight_layout()
plt.show()

## 2. FresnelZoneAperture - 菲涅尔波带片掩膜

菲涅尔波带片是一种二值化的余弦函数掩膜。

In [ ]:
# 创建菲涅尔波带片掩膜
fresnel_zone = FresnelZoneAperture(
    radius=0.56e-3,  # 0.56 mm 半径
    resolution=(256, 256),
    feature_size=1e-5,
    distance_sensor=4e-3,
)

# 绘制掩膜
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fresnel_zone.plot(ax=axes[0])
axes[0].set_title("FresnelZoneAperture - 菲涅尔波带片")

# 显示 PSF
if fresnel_zone.psf is not None:
    psf_rgb = fresnel_zone.psf / fresnel_zone.psf.max()
    axes[1].imshow(psf_rgb)
    axes[1].set_title("对应的 PSF (点扩散函数)")
    axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"掩膜半径: {fresnel_zone.radius * 1e3} mm")
print(f"掩膜形状: {fresnel_zone.mask.shape}")

## 3. MultiLensArray - 多透镜阵列掩膜

多透镜阵列掩膜包含多个随机分布的微透镜。

In [ ]:
# 创建多透镜阵列掩膜
multi_lens = MultiLensArray(
    N=10,  # 10 个微透镜
    resolution=(256, 256),
    feature_size=1e-5,
    distance_sensor=4e-3,
    refractive_index=1.5,
    radius_range=(5e-5, 2e-4),  # 半径范围
    seed=42,  # 随机种子以保证可重复性
    verbose=True,
)

# 绘制掩膜 (高度图)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
multi_lens.plot(ax=axes[0])
axes[0].set_title("MultiLensArray - 多透镜阵列 (高度图)")

# 显示 PSF
if multi_lens.psf is not None:
    psf_rgb = multi_lens.psf / multi_lens.psf.max()
    axes[1].imshow(psf_rgb)
    axes[1].set_title("对应的 PSF (点扩散函数)")
    axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"实际放置的透镜数量: {multi_lens.N}")
print(f"透镜焦距范围: {multi_lens.focal_length.min()*1e3:.3f} - {multi_lens.focal_length.max()*1e3:.3f} mm")

## 4. PhaseContour - 相位轮廓掩膜 (PhlatCam)

### PhaseContour 的详细可视化

展示 PhaseContour 掩膜的生成过程。

In [ ]:
# 创建一个新的 PhaseContour 以展示不同的噪声周期
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

noise_periods = [(8, 8), (16, 16), (32, 32)]

for idx, period in enumerate(noise_periods):
    pc = PhaseContour(
        noise_period=period,
        resolution=(256, 256),
        feature_size=1e-5,
        distance_sensor=4e-3,
        refractive_index=1.5,
        n_iter=10,
    )
    
    # 高度图
    pc.plot(ax=axes[0, idx])
    axes[0, idx].set_title(f"噪声周期: {period}\n高度图")
    
    # 目标 PSF
    axes[1, idx].imshow(pc.target_psf, cmap='gray')
    axes[1, idx].set_title(f"噪声周期: {period}\n目标 PSF (边缘)")
    axes[1, idx].axis('off')

plt.suptitle("PhaseContour - 不同噪声周期的效果", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

相位轮廓掩膜使用 Perlin 噪声和相位恢复算法生成。

In [ ]:
# 创建相位轮廓掩膜
phase_contour = PhaseContour(
    noise_period=(16, 16),  # Perlin 噪声周期
    resolution=(256, 256),
    feature_size=1e-5,
    distance_sensor=4e-3,
    refractive_index=1.5,
    n_iter=10,  # 相位恢复迭代次数
    design_wv=532e-9,  # 设计波长 (绿光)
)

# 绘制掩膜 (高度图)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 高度图
phase_contour.plot(ax=axes[0])
axes[0].set_title("PhaseContour - 相位轮廓 (高度图)")

# 目标 PSF (边缘)
axes[1].imshow(phase_contour.target_psf, cmap='gray')
axes[1].set_title("目标 PSF (边缘检测结果)")
axes[1].axis('off')

# 实际 PSF
if phase_contour.psf is not None:
    psf_rgb = phase_contour.psf / phase_contour.psf.max()
    axes[2].imshow(psf_rgb)
    axes[2].set_title("实际 PSF (点扩散函数)")
    axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"高度图范围: {phase_contour.height_map.min()*1e6:.3f} - {phase_contour.height_map.max()*1e6:.3f} μm")

## 5. RandomBinaryMask - 随机二值掩膜

随机二值掩膜生成伪随机的二值模式，类似于随机散斑图案。

In [ ]:
# 创建随机二值掩膜 (默认 50% 填充率)
random_binary = RandomBinaryMask(
    fill_ratio=0.5,
    resolution=(256, 256),
    feature_size=1e-5,
    distance_sensor=4e-3,
    seed=42,  # 设置随机种子以保证可重复性
)

# 绘制掩膜
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
random_binary.plot(ax=axes[0])
axes[0].set_title("RandomBinaryMask - 随机二值掩膜 (fill_ratio=0.5)")

# 显示 PSF
if random_binary.psf is not None:
    psf_rgb = random_binary.psf / random_binary.psf.max()
    axes[1].imshow(psf_rgb)
    axes[1].set_title("对应的 PSF (点扩散函数)")
    axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"掩膜分辨率: {random_binary.resolution}")
print(f"掩膜形状: {random_binary.mask.shape}")
print(f"填充率 (1的比例): {random_binary.mask.sum() / random_binary.mask.size:.2%}")

In [ ]:
# 展示不同填充率的效果
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

fill_ratios = [0.3, 0.5, 0.7]

for idx, ratio in enumerate(fill_ratios):
    rbm = RandomBinaryMask(
        fill_ratio=ratio,
        resolution=(256, 256),
        feature_size=1e-5,
        distance_sensor=4e-3,
        seed=42,
    )
    
    # 掩膜
    rbm.plot(ax=axes[0, idx])
    axes[0, idx].set_title(f"填充率: {ratio:.1%}")
    
    # PSF
    if rbm.psf is not None:
        psf_rgb = rbm.psf / rbm.psf.max()
        axes[1, idx].imshow(psf_rgb)
        axes[1, idx].set_title(f"填充率: {ratio:.1%}\nPSF")
    axes[1, idx].axis('off')

plt.suptitle("RandomBinaryMask - 不同填充率的效果", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. 所有掩膜对比

将所有掩膜类型并排显示以便比较。

In [ ]:
# 创建一个大的对比图
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

# 第一行: 掩膜本身
masks = [
    (coded_aperture_mls, "CodedAperture\n(MLS)"),
    (coded_aperture_mura, "CodedAperture\n(MURA)"),
    (fresnel_zone, "FresnelZoneAperture"),
    (multi_lens, "MultiLensArray"),
    (random_binary, "RandomBinaryMask"),
]

for idx, (mask_obj, title) in enumerate(masks):
    mask_obj.plot(ax=axes[0, idx])
    axes[0, idx].set_title(title, fontsize=12, fontweight='bold')

# 第二行: PSF
for idx, (mask_obj, title) in enumerate(masks):
    if mask_obj.psf is not None:
        psf_rgb = mask_obj.psf / mask_obj.psf.max()
        axes[1, idx].imshow(psf_rgb)
        axes[1, idx].set_title(f"{title}\nPSF", fontsize=10)
    axes[1, idx].axis('off')

plt.suptitle("所有掩膜类型对比", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 总结

本笔记本演示了 `lensless.hardware.mask` 模块中的所有掩膜类型:

1. **CodedAperture**: 使用 MLS 或 MURA 方法生成的伪随机二值掩膜
2. **FresnelZoneAperture**: 二值化的余弦函数掩膜
3. **MultiLensArray**: 包含多个随机分布的微透镜阵列
4. **PhaseContour**: 基于 Perlin 噪声和相位恢复的相位掩膜
5. **RandomBinaryMask**: 随机生成的二值掩膜，类似于随机散斑图案

每种掩膜都有其独特的光学特性和应用场景。

In [ ]:
import cv2 as cv
import numpy as np
import os

def upsample_binary_mask(input_path, output_path, scale=4):
    """
    上采样二值掩膜图像（保持二值块结构）

    Parameters
    ----------
    input_path : str
        输入 PNG 掩膜路径。
    output_path : str
        输出 PNG 保存路径。
    scale : int
        上采样倍数。例如 scale=4 表示扩大 4 倍。
    """
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"❌ 未找到输入文件：{input_path}")

    # 读取为灰度图
    img = cv.imread(input_path, cv.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"❌ 无法读取图像文件：{input_path}")

    print(f"📥 原始尺寸: {img.shape[1]}x{img.shape[0]}")

    # 上采样（最近邻保持块状）
    upsampled = cv.resize(
        img,
        (img.shape[1] * scale, img.shape[0] * scale),
        interpolation=cv.INTER_NEAREST
    )

    # 保持二值化（避免插值带来的灰值）
    _, binary_mask = cv.threshold(upsampled, 127, 255, cv.THRESH_BINARY)

    # 保存结果
    cv.imwrite(output_path, binary_mask)
    print(f"✅ 已保存上采样结果: {output_path}")
    print(f"📤 新尺寸: {binary_mask.shape[1]}x{binary_mask.shape[0]}")

    return binary_mask


# 举例
input_path = "/Users/qiujinyu/Computational_Imaging/无透镜相关代码仓库/SLMImagingPipeline/data/mask_patten_gen/random_resolution=64x48_fill_ratio=0.6.png"
output_path = "/Users/qiujinyu/Computational_Imaging/无透镜相关代码仓库/SLMImagingPipeline/data/mask_patten_gen/random_upsampled_16x.png"

upsample_binary_mask(input_path, output_path, scale=16)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2 as cv

def generate_radial_mask(resolution=(512, 512), n_sectors=20, random=False, seed=None):
    """
    生成径向二值掩膜。
    
    Parameters
    ----------
    resolution : tuple(int, int)
        输出图像分辨率 (height, width)
    n_sectors : int
        径向扇区数量，例如 20、40、60。
    random : bool
        是否随机扰动每个扇区的角度。
    seed : int
        随机种子（仅在 random=True 时有效）
    
    Returns
    -------
    mask : np.ndarray
        二值掩膜，uint8 格式，0/255。
    """
    h, w = resolution
    cx, cy = w // 2, h // 2

    # 坐标网格
    y, x = np.indices((h, w))
    x = x - cx
    y = cy - y  # 翻转 y 方向以便显示正确
    angles = np.arctan2(y, x)  # [-π, π]

    # 随机扰动角度边界
    if random:
        rng = np.random.default_rng(seed)
        offsets = rng.uniform(-np.pi / n_sectors / 2, np.pi / n_sectors / 2, size=n_sectors)
    else:
        offsets = np.zeros(n_sectors)

    # 确定扇区边界
    mask = np.zeros_like(angles, dtype=np.uint8)
    for i in range(n_sectors):
        theta_start = -np.pi + i * 2 * np.pi / n_sectors + offsets[i]
        theta_end = -np.pi + (i + 1) * 2 * np.pi / n_sectors + offsets[i]
        region = (angles >= theta_start) & (angles < theta_end)
        mask[region] = 255 if i % 2 == 0 else 0

    return mask

def save_radial_mask(mask, out_dir, resolution, n_sectors, random=False, seed=None):
    """保存掩膜为 PNG，自动生成文件名"""
    os.makedirs(out_dir, exist_ok=True)

    parts = [f"radial"]
    if random:
        parts.append("random")
    else:
        parts.append("periodic")

    parts.append(f"sectors={n_sectors}")
    parts.append(f"res={resolution[1]}x{resolution[0]}")
    if random and seed is not None:
        parts.append(f"seed={seed}")

    filename = "_".join(parts) + ".png"
    save_path = os.path.join(out_dir, filename)

    cv.imwrite(save_path, mask)
    print(f"✅ 已保存掩膜: {save_path}")


# 输出路径（修改成你的路径）
out_dir = "/Users/qiujinyu/Computational_Imaging/无透镜相关代码仓库/SLMImagingPipeline/data/mask_patten_gen"
# 周期性掩膜示例
for n in [20, 40, 60]:
    mask = generate_radial_mask(resolution=(768, 1024), n_sectors=n, random=False)
    save_radial_mask(mask, out_dir, (768, 1024), n, random=False)
# 随机掩膜示例
mask_rand = generate_radial_mask(resolution=(768, 1024), n_sectors=30, random=True, seed=42)
save_radial_mask(mask_rand, out_dir, (768, 1024), 30, random=True, seed=42)


In [ ]:
import matplotlib.pyplot as plt

noise_periods = (16, 16)
pc = PhaseContour(
    noise_period=noise_periods,
    resolution=(256, 256),
    feature_size=1e-5,
    distance_sensor=4e-3,
    refractive_index=1.5,
    n_iter=10,
)

masks = [
    (coded_aperture_mls, "编码孔径\n(MLS)"),
    (coded_aperture_mura, "编码孔径\n(MURA)"),
    (fresnel_zone, "FZA"),
    (pc,"Phlatcam 的高度图"),
    (multi_lens, "微透镜"),
    (random_binary, "随机二值掩膜"),
    (mask_rand, "径向掩膜"),
]

# 假设你已有以下对象（请确保定义好）
# coded_aperture_mls, coded_aperture_mura, fresnel_zone, multi_lens, random_binary, mask_rand
# 并指定外部图像路径
output_path = "/Users/qiujinyu/Computational_Imaging/无透镜相关代码仓库/SLMImagingPipeline/data/mask_patten_gen/random_upsampled_16x.png"

# 读取外部图像
if os.path.exists(output_path):
    external_mask = cv.imread(output_path, cv.IMREAD_GRAYSCALE)
    print(f"✅ 已读取外部图像: {output_path}")
else:
    raise FileNotFoundError(f"❌ 文件不存在: {output_path}")

# 创建一个单行的掩膜对比图（共7张）
fig, axes = plt.subplots(1, 8, figsize=(24, 4))

masks = [
    (coded_aperture_mls, "编码孔径\n(MLS)"),
    (coded_aperture_mura, "编码孔径\n(MURA)"),
    (fresnel_zone, "FZA"),
    (pc,"Phlatcam 的高度图"),
    (multi_lens, "微透镜"),
    (random_binary, "随机二值掩膜"),
    (external_mask, "随机二值掩膜（上采样）"),
    (mask_rand, "径向掩膜"),
    
]

# 显示掩膜
for idx, (mask_obj, title) in enumerate(masks):
    ax = axes[idx]
    if hasattr(mask_obj, "plot"):  # 如果是类对象
        mask_obj.plot(ax=ax)
    elif isinstance(mask_obj, np.ndarray):  # numpy 数组（如 mask_rand 或外部PNG）
        ax.imshow(mask_obj, cmap="gray", vmin=0, vmax=255)
        ax.set_xlabel("[px]")
        ax.set_ylabel("[px]")
    else:
        raise TypeError(f"不支持的掩膜类型: {type(mask_obj)}")

    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.axis("off")

plt.suptitle("掩膜类型与导入图像对比", fontsize=16, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()